# End-to-end verification: query the pipeline output with DuckDB

This notebook reads the Parquet files written by the pipeline directly from
MinIO/S3 and runs analytical queries over them. It's the smoke test that
proves the system works end-to-end — ingestion → quality → S3 → query.

**Prereqs:** `docker compose up --build` is running (MinIO on :9000, pipeline
ingesting from Bluesky firehose or demo mode).

```bash
pip install duckdb
```


In [18]:
import duckdb

con = duckdb.connect()
con.execute("""
    INSTALL httpfs;
    LOAD httpfs;
    SET s3_endpoint='localhost:9000';
    SET s3_access_key_id='minioadmin';
    SET s3_secret_access_key='minioadmin';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
""")

# `union_by_name=true` makes the read robust to schema evolution:
# v1 wrote `subreddit`, v2 writes `source`. Without union_by_name, DuckDB
# rejects mixed-schema reads. With it, missing columns are filled with NULL.
df = con.execute("""
    SELECT ticker,
           COUNT(*)                              AS mentions,
           ROUND(AVG(sentiment_compound), 3)     AS avg_sentiment,
           ROUND(STDDEV(sentiment_compound), 3)  AS sentiment_stddev
    FROM read_parquet('s3://social-data/mentions/**/*.parquet',
                      union_by_name=true)
    GROUP BY ticker
    ORDER BY mentions DESC
    LIMIT 5;
""").fetchdf()

print(df)


  ticker  mentions  avg_sentiment  sentiment_stddev
0   NVDA       316          0.339             0.394
1   TSLA       217          0.069             0.072
2   MSFT       206          0.288             0.405
3    SPY       192         -0.142             0.148
4   AAPL       116          0.599             0.000


In [19]:
# Volume + quality breakdown by source.
# Confirms which sources are landing data and that quality scoring is
# producing a real distribution (not all 1.0).
df = con.execute("""
    SELECT source,
           COUNT(*)                              AS posts,
           ROUND(AVG(quality_score), 3)          AS avg_quality,
           ROUND(MIN(quality_score), 3)          AS min_quality,
           SUM(CASE WHEN tickers IS NOT NULL AND len(tickers) > 0 THEN 1 ELSE 0 END)
                                                 AS posts_with_tickers
    FROM read_parquet('s3://social-data/posts/**/*.parquet',
                      union_by_name=true)
    GROUP BY source;
""").fetchdf()

print(df)


  source  posts  avg_quality  min_quality  posts_with_tickers
0   demo   2000          1.0          1.0              2000.0


In [15]:
# Most recent 10 posts mentioning a specific ticker — the kind of query
# a model trainer would use to fetch the per-ticker time series.
df = con.execute("""
    SELECT to_timestamp(created_utc) AS ts,
           ticker,
           sentiment_compound,
           source,
           post_id
    FROM read_parquet('s3://social-data/mentions/**/*.parquet',
                      union_by_name=true)
    WHERE ticker = 'NVDA'
    ORDER BY created_utc DESC
    LIMIT 10;
""").fetchdf()

print(df)


                                ts ticker  sentiment_compound source  \
0 2026-05-01 13:36:00.479274-07:00   NVDA             -0.2023   demo   
1 2026-05-01 13:35:56.966214-07:00   NVDA              0.6808   demo   
2 2026-05-01 13:35:53.946763-07:00   NVDA              0.5574   demo   
3 2026-05-01 13:35:47.915814-07:00   NVDA              0.5574   demo   
4 2026-05-01 13:35:30.305747-07:00   NVDA              0.5574   demo   
5 2026-05-01 13:35:29.303680-07:00   NVDA              0.6808   demo   
6 2026-05-01 13:35:28.289318-07:00   NVDA              0.5574   demo   
7 2026-05-01 13:35:19.223666-07:00   NVDA              0.6808   demo   
8 2026-05-01 13:35:18.722292-07:00   NVDA             -0.2023   demo   
9 2026-05-01 13:35:17.199853-07:00   NVDA             -0.2023   demo   

     post_id  
0  demo_1935  
1  demo_1928  
2  demo_1922  
3  demo_1910  
4  demo_1875  
5  demo_1873  
6  demo_1871  
7  demo_1853  
8  demo_1852  
9  demo_1849  


In [16]:
# Hourly mention volume — repeating the date_trunc expression in GROUP BY
# (instead of `GROUP BY hour`) avoids DuckDB's strict alias-resolution rule.
df = con.execute("""
    SELECT date_trunc('hour', to_timestamp(created_utc)) AS hour,
           COUNT(*)                                       AS mentions,
           COUNT(DISTINCT ticker)                         AS distinct_tickers
    FROM read_parquet('s3://social-data/mentions/**/*.parquet',
                      union_by_name=true)
    GROUP BY date_trunc('hour', to_timestamp(created_utc))
    ORDER BY hour DESC
    LIMIT 24;
""").fetchdf()

print(df)


                       hour  mentions  distinct_tickers
0 2026-05-01 13:00:00-07:00      2359                19


In [17]:
# Sanity check: extreme-sentiment posts. Useful for spot-checking that
# VADER scoring is reasonable on real text (which is messier than the
# demo templates).
df = con.execute("""
    SELECT source,
           ROUND(sentiment_compound, 2) AS s,
           tickers,
           SUBSTR(body, 1, 120)         AS preview
    FROM read_parquet('s3://social-data/posts/**/*.parquet',
                      union_by_name=true)
    WHERE ABS(sentiment_compound) > 0.7
      AND len(tickers) > 0
    ORDER BY ABS(sentiment_compound) DESC
    LIMIT 10;
""").fetchdf()

print(df.to_string())


Empty DataFrame
Columns: [source, s, tickers, preview]
Index: []
